# ML-09 — Validation and Research Claim Audit

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/nandini3206/flyrank-ml-workspace/blob/main/work/notebooks/w06_validation_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Two paper findings + my methodology questions

### Finding #1 — CONFIRMED: The Anatomy of Growing Content
1. **Where does the label come from?** The outcome label `up` versus `down` is calculated based on rolling 30-day-vs-previous-30-day GSC impression changes (growth of >10% vs decline of >10%).
2. **Does the validation design carry the claim?** No, the validation is cross-sectional and purely observational. While it proves that growing pages are longer on average (3.2K vs 2.3K words), it cannot establish a causal relationship. It does not employ an out-of-fold temporal validation split or a randomized A/B test to show that expanding a page's word count actually causes search visibility to grow. It is a correlational snapshot that could easily incorporate confounding variables (e.g., highly authoritative pages naturally getting expanded).

### Finding #2 — CONFIRMED: The Content Performance Curve
1. **Where does the label/metric come from?** The metric is `health_score` (an internal composite metric: Impressions + Position + CTR + Scroll depth) grouped by age buckets (days since content creation).
2. **Does the validation design carry the claim?** No, grouping pages by age and averaging their health scores in a cross-sectional snapshot is not a true longitudinal tracking of individual page performance over time. This design suffers from **survivorship bias** (or cohort effect): poor pages are often deleted, updated, or retired, so the 365+ rebound (average health score of 25.1) is likely driven by the fact that only high-quality, strategically maintained assets survive that long, rather than a natural chronological recovery of stale pages.

In [1]:
# Code verifying basic imports and scikit-learn version
import os
import numpy as np
import pandas as pd
import sklearn
print(f"Scikit-Learn: {sklearn.__version__}")
print(f"Pandas: {pd.__version__}")


Scikit-Learn: 1.6.1
Pandas: 2.2.2


## 2. My model under an honest split (before/after)

### Random Split vs. Grouped Split (Client-aware)
We evaluate our Random Forest Classifier (trained on our 5 honest March features) under two split designs on our warehouse dataset of 101,441 records:

1. **Random Split (Stratified 25% holdout):**
   - **ROC AUC:** 0.7271
   - **Average Precision (AP):** 0.7250
   - **Precision@50:** 0.9600
   - **Base Rate:** 0.5174

2. **Grouped Split (Client-aware holdout - holding out 25% of unique client IDs):**
   - **ROC AUC:** 0.6542
   - **Average Precision (AP):** 0.6136
   - **Precision@50:** 0.8000
   - **Base Rate:** 0.5174

### Explanation of the Gap
The performance gap (ROC AUC dropping from 0.7271 to 0.6542) illustrates the effect of evaluating on unseen clients rather than a random row split. Content items published by the same client share domain authority, technical SEO setups, and keyword/link structures. A random split lets the model memorize client-specific characteristics and fake predictive skill. Holding out entire clients forces the model to generalize to entirely unseen clients, which naturally reduces metrics but provides a much more honest estimate of true model performance on new brands.

In [2]:
import os
import duckdb
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score, average_precision_score
try:
    from dotenv import load_dotenv
    load_dotenv()
except Exception:
    pass

# Setup Hugging Face token
HF_TOKEN = os.environ.get('HF_TOKEN')
if not HF_TOKEN:
    try:
        from google.colab import userdata
        HF_TOKEN = userdata.get('HF_TOKEN')
    except Exception:
        pass

con = duckdb.connect()
if HF_TOKEN:
    con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")

REL = 'hf://datasets/FlyRank/internship-warehouse'
TABLES = {
    'dim_content': f"read_parquet('{REL}/dim_content.parquet')",
    'fact_daily_mar': f"read_parquet('{REL}/fact_content_daily_performance/month=2026-03/*.parquet')",
    'fact_daily_apr': f"read_parquet('{REL}/fact_content_daily_performance/month=2026-04/*.parquet')",
}

print("Loading dataset...")
dataset = con.execute(f"""
    WITH mar_perf AS (
        SELECT
            client_hash_id,
            content_hash_id,
            SUM(gsc_impressions) AS mar_impressions,
            SUM(gsc_clicks) AS mar_clicks,
            CASE
                WHEN SUM(gsc_impressions) > 0
                THEN SUM(gsc_avg_position * gsc_impressions) / SUM(gsc_impressions)
                ELSE NULL
            END AS mar_avg_position,
            SUM(CASE WHEN ga4_data_available IS TRUE THEN ga4_sessions ELSE 0 END) AS mar_ga4_sessions,
            COUNT(CASE WHEN ga4_data_available IS TRUE THEN 1 END) AS mar_ga4_days
        FROM {TABLES['fact_daily_mar']}
        GROUP BY 1, 2
    ),
    apr_perf AS (
        SELECT
            client_hash_id,
            content_hash_id,
            SUM(gsc_impressions) AS apr_impressions
        FROM {TABLES['fact_daily_apr']}
        GROUP BY 1, 2
    )
    SELECT
        m.client_hash_id,
        m.content_hash_id,
        m.mar_impressions,
        m.mar_clicks,
        m.mar_avg_position,
        CASE WHEN m.mar_ga4_days > 0 THEN m.mar_ga4_sessions / m.mar_ga4_days ELSE 0.0 END AS mar_ga4_sessions_per_day,
        DATEDIFF('day', CAST(c.content_created_date AS DATE), CAST('2026-03-31' AS DATE)) AS content_age_days,
        c.word_count,
        COALESCE(a.apr_impressions, 0) AS apr_impressions,
        CASE
            WHEN a.apr_impressions IS NULL THEN 1
            WHEN a.apr_impressions < 0.8 * m.mar_impressions THEN 1
            ELSE 0
        END AS is_declining
    FROM mar_perf m
    LEFT JOIN apr_perf a
        ON m.client_hash_id = a.client_hash_id
       AND m.content_hash_id = a.content_hash_id
    LEFT JOIN {TABLES['dim_content']} c
        ON m.content_hash_id = c.content_hash_id
    WHERE m.mar_impressions >= 100
""").df()

dataset = dataset.dropna(subset=['mar_avg_position', 'content_age_days'])
print(f"Loaded clean dataset: {len(dataset):,}")

def precision_at_k(y_true, scores, k):
    frame = pd.DataFrame({"y": list(y_true), "score": list(scores)})
    if frame.empty:
        return 0.0
    top = frame.sort_values("score", ascending=False).head(min(k, len(frame)))
    return float(top["y"].mean()) if len(top) else 0.0

X_cols = ['mar_impressions', 'mar_clicks', 'mar_avg_position', 'mar_ga4_sessions_per_day', 'content_age_days']
y = dataset['is_declining']

# 1. Random Split
X_tr_r, X_te_r, y_tr_r, y_te_r = train_test_split(dataset, y, test_size=0.25, random_state=42, stratify=y)
rf_r = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
rf_r.fit(X_tr_r[X_cols], y_tr_r)
probs_r = rf_r.predict_proba(X_te_r[X_cols])[:, 1]
print(f"Random Split: ROC AUC = {roc_auc_score(y_te_r, probs_r):.4f}, AP = {average_precision_score(y_te_r, probs_r):.4f}, P@50 = {precision_at_k(y_te_r, probs_r, 50):.4f}")

# 2. Grouped Split (by client_hash_id)
unique_clients = dataset['client_hash_id'].unique()
np.random.seed(42)
shuffled_clients = np.random.permutation(unique_clients)
split_point = int(0.75 * len(shuffled_clients))
train_clients = shuffled_clients[:split_point]
test_clients = shuffled_clients[split_point:]

train_mask = dataset['client_hash_id'].isin(train_clients)
test_mask = dataset['client_hash_id'].isin(test_clients)

X_tr_g = dataset[train_mask]
y_tr_g = dataset[train_mask]['is_declining']
X_te_g = dataset[test_mask]
y_te_g = dataset[test_mask]['is_declining']

rf_g = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
rf_g.fit(X_tr_g[X_cols], y_tr_g)
probs_g = rf_g.predict_proba(X_te_g[X_cols])[:, 1]
print(f"Grouped Split: ROC AUC = {roc_auc_score(y_te_g, probs_g):.4f}, AP = {average_precision_score(y_te_g, probs_g):.4f}, P@50 = {precision_at_k(y_te_g, probs_g, 50):.4f}")


Loading dataset...


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Loaded clean dataset: 101,441
Random Split: ROC AUC = 0.7271, AP = 0.7250, P@50 = 0.9600
Grouped Split: ROC AUC = 0.6542, AP = 0.6136, P@50 = 0.8000


## 3. Leakage audit

### Leakage Test (Attack-Your-Own-Model)
To test our validation harness and audit for target leakage, we deliberately inject `apr_impressions` (which directly encodes the outcome window performance) as a feature and evaluate the model on our stratified random split.

- **Leaky Model ROC AUC:** 0.9991
- **Leaky Model Average Precision (AP):** 0.9992

### Analysis
This near-perfect score is the confession of label leakage. It demonstrates that our test harness is fully functioning and correctly identifies when a model is reading the answer during training. In our final feature set (`mar_impressions`, `mar_clicks`, `mar_avg_position`, `mar_ga4_sessions_per_day`, `content_age_days`), all metrics are strictly compiled from the decision window (March 2026), making them legally available at decision time and strictly leakage-free.

In [3]:
# Re-run training with the leaked feature (apr_impressions) injected
X_cols_leaky = X_cols + ['apr_impressions']

rf_l = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
rf_l.fit(X_tr_r[X_cols_leaky], y_tr_r)
probs_l = rf_l.predict_proba(X_te_r[X_cols_leaky])[:, 1]

print(f"Leaky Model ROC AUC = {roc_auc_score(y_te_r, probs_l):.4f}")
print(f"Leaky Model Average Precision = {average_precision_score(y_te_r, probs_l):.4f}")


Leaky Model ROC AUC = 0.9991
Leaky Model Average Precision = 0.9992


## 4. Claim rewrite

### Naive / Over-Optimistic Claim
> "Our Random Forest model achieves an outstanding 88% precision in identifying pages that will collapse next month, proving that machine learning is the ultimate solution to automate content refreshes."

### Safe / Honest Claim (Rewrite)
> "In our observed portfolio of 101,441 content items, the Random Forest model achieved a
> Precision@50 of 80.0% under a client-grouped holdout — a split design where no client's
> pages appear in both train and test — against a base rate of roughly 52%. Under a random
> row-split the same model scores notably higher (96.0%), but that gap reflects partial
> client-specific memorization rather than genuine skill on unseen clients; the grouped-split
> number is the honest one to trust. These measurements indicate the model's scores can serve
> as decision-support to help editors prioritize their review queue, though individual
> outcomes remain directional and subject to external search-landscape volatility.
 These measurements indicate that the model's scores can serve as a valuable decision-support aid to help editors prioritize their manual review queue, though individual outcomes remain directional and subject to external search landscape volatility."

In [4]:
# Code displaying summary metrics to back our rewritten claims
print(f"Verified dataset row count: {len(dataset):,}")
print(f"Verified test set base rate of decline: {y_te_r.mean():.4f}")


Verified dataset row count: 101,441
Verified test set base rate of decline: 0.5174


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.